# Input - Output

## String

### Split String

```cpp
std::vector<std::string> split(const std::string& s, char delim) {
    std::vector<std::string> answers;

    std::stringstream ss(s);

    // keep '' tokens if e.g 'a,,b' -> ['a', '', 'b']
    std::string token;
    while(std::getline(ss, token, delim)) {
        answers.emplace_back(token);
    }

    return answers;
}
```

### Trim String

```cpp
bool is_space(char c) { return c == ' ' || c == '\t'; }

void trim(std::string& s) {

    // left trim
    std::size_t start = 0;
    while (start < s.size() && is_space(s[start])) ++start;

    // right trim
    std::size_t end = s.size();
    while (end > start && is_space(s[end - 1])) --end;

    s = s.substr(start, end - start);
}
```

## File

### Read file, line by line

```cpp
void parse_csv(const std::string& filename, std::vector<Course>& courses) {
    std::ifstream in_stream(filename);
    if (!in_stream) return;
    
    std::string line;
    std::getline(in_stream, line);            // read header

    while (std::getline(in_stream, line)) {  // read the rest
        auto elements = // write your 'split' function
        // Do something
    }

    return;    
}
```

### Write file, line by line

```cpp
void write_courses(std::vector<Course> courses) {
    std::ofstream out_stream(PATH_STR);
    if (!out_stream) return;

    // Write header
    out_stream << "Title,Number of Units,Quarter" << '\n';

    // Write the rest
    for (const auto& course : courses) {
        out_stream << course << '\n';
    }

    return;
}
```

# Function

### Return Type

##### `T&` vs `T`

| Aspect      | Choice | Why                                |
|:---|:---:|:---|
| `T&` vs `T`     | `T&`    | If you want chained-same operations.<br> For example: `std::cout << 'abc' << 123;`  |
|             | `T`     | Return a new value / independent    |

##### `std::optional<T>` vs `std::variant<T1, ...>`

| Aspect | `std::optional<T>` | `std::variant<Ts...>` |
| :---: | :---: | :---: |
| Meaning | Maybe one `T`, or no value (`nullopt`) | Exactly one of several types |
| Access | `has_value`, `value`, `*`, `value_or` | `std::visit`, `std::get`, `holds_alternative`, `index()` |
| Use when | Presence of one type is optional | You need one-of-many types |

### Handling `std::optional<T>`

1. (C++23) we can use monadic calls which is like chained function call in Python.

- For fallback, use `.value_or("... not found")`
```cpp
    std::string out = course.value_or("Course not found.");
```


- If you want to transform `std::optional<T>` -> `std::optional<U>`, 

    applies `.transform(lambda func)` before `.value_or()`

```cpp
    std::string out = course.transform(
        // map std:optional<Course> -> std::optimal<string>
        [](const Course& c) { return "Found: " + c.title; }
    )
```


- Remember to compile by: `g++ -std=c++23 main.cpp -o main`

2. (older C++) just use if-else:

```cpp
    auto course = db.find_course(title);

    if (course) {   // or: if (course.has_value())
        std::cout << course->title << "\n";
    } else {
        std::cout << "Course not found.\n";
    }
```

# Pointers

### `unique_ptr` vs `shared_ptr`

| Aspect          | `unique_ptr` (one owner)             | `shared_ptr` (many owners)          |
|:-----------------:|:--------------------------------------:|:-------------------------------------:|
| Who owns it     | Only one pointer                     | Many pointers share the same thing  |
| Copying         | You can move it; CANNOT copy it  | You can move it AND copy it                     |
| Cleanup time    | When that one owner goes away        | When the last shared owner is gone  |
| Extra cost      | Almost none                          | Extra cost to count owners          |
| Good for        | One clear lifetime                   | Sharing across places               |
| Example         | Tree or list child pointers          | Shared cache / listeners / async task   |

### `std::string_view` vs `std::string*`

| Aspect | `std::string_view` | `std::string*` |
| :---: |:------: | :----------------: |
| Meaning | Just a pointer and a size (or two pointers) | Implies string ends<br> in null char `'\0'` |
| Null Pointer | Can't have `nullptr` | Has `nullptr` |
| Mutability | Read-only; cannot modify underlying text | Can modify |
| API | Read-only inspection/slicing:<br>`size/empty`, `operator[]`, `front/back`, `data`, `find/rfind`, `find_first/last_of`, `substr`, `starts_with/ends_with`, `remove_prefix/suffix`; no mutating | Full `std::string` API<br>(`length`, `capacity`, `push_back`, `append`, `insert`, `erase`, `resize`, etc.) |
| Copy cost | Tiny, cheap to copy | Tiny, cheap to copy |

# Class

### `explicit` and automatic type conversion

`explicit` helps avoid automatic type conversion,

```cpp
    explicit ClassName(T* p) : ptr{p} {}
```

because: any non-`explicit` class that has a constructor taking an argument of another class will have automatic conversion when the equals sign appears, left-hand side having the class name, and the right-hand side having the argument type.

For example, if:
```cpp
    ClassName(int p) : ptr{p} {}
    ClassName(int* p) : ptr{p} {}
```

Then these will do automatic type conversation:
```cpp
    ClassName x = 5;            // equivalent to `ClassName(5);`
    ClassName x = new int{5};   // equivalent to 'ClassName(new int{5});
```

### Copy Constructor vs Copy Assignment Operator

1. Both are 2 different memory address, we just dont waste another copy of returned value. 

2. Rule of 3: any of these implemented, we want all.
- Deferencing
- Copy Constructor
- Copy Assignment Operator

```cpp
class X {
    public:
        X() = default;                    // let compiler generate code in compile time
        X(const X& other) {               // copy constructor
            // buid this from 'other'
        }
        X& operator=(const X& other) {    // copy assignment
            // replace this object's fields with 'other'
            return *this;
        }
};
```

Copy constructor:
```cpp
    X a;
    X b = a;   // copy constructor
    X c(a);    // copy constructor
```
Here b and c are being created for the first time.

Copy assignment operator:
```cpp
    X a;
    X b;
    X c;
    b = a = c;     // copy assignment: (b = a) = c
```
Here b already exists, so it is not being constructed again. It is being assigned to.

# Algo

### Swap

1. Swap big containers (e.g `std::string`, `std::vector`)

```cpp
    void swap(T& a, T& b) {
        T temp = std::move(a);
        a = std::move(b);
        b = std::move(temp);
    }
```

2. Swap small stuff (e.g pointer, `int`), `std::move()` is not boosting

```cpp
    void swap(T& a, T& b) {
        T temp = a;
        a = b;
        b = temp;
    }
```

# -